# Update 05 - Time coverage of the campaign dataset

**Purpose:** Characterise the temporal structure of the processed dataset (`data/processed/full_data.csv`, N = 145,784) for the manuscript and SM:

monthly valid-observation days and retained counts, median cadence and longest gap within each month, observed ranges, and the fraction of observations within the Mathar validity domain ($T \in [10, 25]$ °C). Also computes the retained-observation mean and the time-weighted campaign mean of the environmental parameters, which in general differ because the data are not uniformly sampled in time.

**Context (manuscript revision):** earlier text stated the campaign covered "January - August 2025" with "natural diurnal and seasonal variations", which may be read as continuous operation. The dataset is in fact a set of 104 contiguous observation segments spanning 232 calendar days with an overall
duty cycle near 9%. This notebook provides the numbers to describe the coverage honestly.

**Expected key numbers (verified):**

| Quantity | Value |
|---|---|
| Total observations | 145,784 |
| Calendar span | 2025-01-01 … 2025-08-20 (232 days) |
| Active observation days | 89 |
| Continuous segments (>2 h gap split) | 104 |
| Median segment length | ~5.7 h |
| Median intra-day covered span | ~12.1 h |
| Longest data gap | ~111 days (Feb 25 - Jun 16) |
| Median sampling interval | ~12.9 s |
| Duty cycle (vs 232 days) | ~9.4 % |

In [1]:
import pandas as pd
import numpy as np
import json, os

# Load data
DATA = '../../data/processed/full_data.csv'
df = pd.read_csv(DATA, encoding='utf-8-sig')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').reset_index(drop=True)
N = len(df)
print(f"N = {N}")
print(f"time range: {df['time'].iloc[0]}  ->  {df['time'].iloc[-1]}")
print(f"calendar span: {(df['time'].iloc[-1]-df['time'].iloc[0]).days} days")

# months present
df['month'] = df['time'].dt.to_period('M')
print("\nmonths present:", sorted(df['month'].unique().astype(str)))

N = 145784
time range: 2025-01-01 00:06:26.245654+00:00  ->  2025-08-20 20:37:37.682980+00:00
calendar span: 231 days

months present: ['2025-01', '2025-02', '2025-06', '2025-07', '2025-08']


/var/folders/qw/t4bxrq7902q6yx_4ncnk_c_40000gn/T/ipykernel_42624/490405800.py:18: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['time'].dt.to_period('M')


In [2]:
# Per-month coverage table
rows = []
for month, g in df.groupby('month'):
    t = g['time'].values.astype('int64') / 1e9   # epoch seconds
    active_days = g['time'].dt.date.nunique()
    # sampling intervals within month (positive)
    dt = np.diff(t)
    dt = dt[dt > 0]
    med_cad = np.median(dt) if len(dt) else np.nan
    max_gap = dt.max() if len(dt) else np.nan
    rows.append({
        'month': str(month),
        'n_obs': len(g),
        'active_days': active_days,
        'median_cadence_s': med_cad,
        'longest_gap_d': max_gap / 86400 if not np.isnan(max_gap) else np.nan,
        'T_range': f"{g['temperature'].min():.1f}-{g['temperature'].max():.1f}",
        'RH_range': f"{g['humidity'].min():.1f}-{g['humidity'].max():.1f}",
        'P_range': f"{g['pressure'].min():.1f}-{g['pressure'].max():.1f}",
        'frac_T_in_Mathar_domain': float(((g['temperature'] >= 10) & (g['temperature'] <= 25)).mean()),
    })

cov = pd.DataFrame(rows)
print("=== Monthly coverage table ===")
print(cov.to_string(index=False))

# overall
dt_all = np.diff(df['time'].values.astype('int64') / 1e9)
dt_all = dt_all[dt_all > 0]
print("\n=== Overall ===")
print(f"active days: {df['time'].dt.date.nunique()}")
print(f"median cadence: {np.median(dt_all):.1f} s")
print(f"longest gap: {dt_all.max()/86400:.1f} days")
print(f"fraction with T > 25 C: {((df['temperature']>25).mean()*100):.1f} %")
print(f"fraction with T in [10,25] C: {(((df['temperature']>=10)&(df['temperature']<=25)).mean()*100):.1f} %")

=== Monthly coverage table ===
  month  n_obs  active_days  median_cadence_s  longest_gap_d   T_range  RH_range      P_range  frac_T_in_Mathar_domain
2025-01  12109           17         13.427357       4.491913 17.5-21.4 28.5-40.9 964.6-1006.8                      1.0
2025-02     25           13      41317.220724       5.895223 20.2-22.1 28.5-38.4  982.2-999.4                      1.0
2025-06  28911           13         12.849338       1.464441 29.3-33.3 22.8-34.6  980.6-995.7                      0.0
2025-07  54397           26         12.759417       3.982549 25.9-34.6 18.9-44.0  974.1-993.3                      0.0
2025-08  50342           20         12.920514       0.993684 25.4-31.9 24.8-45.3  978.9-994.5                      0.0

=== Overall ===
active days: 89
median cadence: 12.9 s
longest gap: 111.3 days
fraction with T > 25 C: 91.7 %
fraction with T in [10,25] C: 8.3 %


In [3]:
# Continuous observation segments (split on gaps > 2 h)
t = df['time'].values.astype('int64') / 1e9
seg_bounds = [0]
for i in range(1, N):
    if t[i] - t[i-1] > 2 * 3600:
        seg_bounds.append(i)
seg_bounds.append(N)
seg_lens_h = []
for s in range(len(seg_bounds)-1):
    i0, i1 = seg_bounds[s], seg_bounds[s+1]
    seg_lens_h.append((t[i1-1] - t[i0]) / 3600)
seg_lens_h = np.array(seg_lens_h)
print(f"number of segments (>2 h gap split): {len(seg_lens_h)}")
print(f"segment length h: min={seg_lens_h.min():.1f} median={np.median(seg_lens_h):.1f} max={seg_lens_h.max():.1f}")
print(f"total active time: {seg_lens_h.sum()/24:.1f} days")
duty = seg_lens_h.sum() / ((t[-1]-t[0])/3600) * 100
print(f"duty cycle: {duty:.1f} %")

number of segments (>2 h gap split): 104
segment length h: min=0.0 median=5.6 max=98.8
total active time: 35.6 days
duty cycle: 15.3 %


In [4]:
# Retained-observation mean vs time-weighted mean

# simple mean over retained observations
simple = {
    'T': df['temperature'].mean(),
    'RH': df['humidity'].mean(),
    'P': df['pressure'].mean(),
}

# time-weighted mean: weight each observation by half the distance to the
# neighbouring samples (trapezoidal / Voronoi weights on the time axis)
tt = df['time'].values.astype('int64') / 1e9
w = np.empty(N)
w[0] = tt[1] - tt[0]
w[-1] = tt[-1] - tt[-2]
for i in range(1, N-1):
    w[i] = 0.5 * (tt[i+1] - tt[i-1])
w = w / w.sum()

tw = {
    'T': float((df['temperature'].values * w).sum()),
    'RH': float((df['humidity'].values * w).sum()),
    'P': float((df['pressure'].values * w).sum()),
}

print("=== Mean conditions: retained-observation vs time-weighted ===")
print(f"  retained-observation:  T = {simple['T']:.2f} C, RH = {simple['RH']:.2f} %, P = {simple['P']:.1f} hPa")
print(f"  time-weighted:         T = {tw['T']:.2f} C, RH = {tw['RH']:.2f} %, P = {tw['P']:.1f} hPa")
print(f"  difference:            dT = {tw['T']-simple['T']:+.2f} C, dRH = {tw['RH']-simple['RH']:+.2f} %")

# save
out = {
    'N': N,
    'span_days': float((tt[-1]-tt[0])/86400),
    'active_days': int(df['time'].dt.date.nunique()),
    'n_segments': int(len(seg_lens_h)),
    'duty_cycle_pct': float(duty),
    'mean_retained': simple,
    'mean_timeweighted': tw,
    'monthly': cov.to_dict('records'),
}
with open('campaign_time_coverage.json', 'w') as f:
    json.dump(out, f, indent=1, default=str)
print("\nsaved campaign_time_coverage.json")

=== Mean conditions: retained-observation vs time-weighted ===
  retained-observation:  T = 28.87 C, RH = 30.23 %, P = 986.4 hPa
  time-weighted:         T = 25.64 C, RH = 31.04 %, P = 987.9 hPa
  difference:            dT = -3.23 C, dRH = +0.81 %

saved campaign_time_coverage.json
